In [15]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langsmith import Client
from langchain.agents import create_react_agent, AgentExecutor

import warnings
warnings.filterwarnings("ignore")

## Load environment variables


In [2]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

## Search tool

In [4]:
search_tool = TavilySearchResults(max_results=2)

result = search_tool.invoke("What is the capital of India?")
result

[{'url': 'https://www.mappr.co/capital-cities/india',
  'content': 'Skip to content\n\nmappr logo\n\nmappr logo\n\n# What is the Capital of India?\n\nNew Delhi is the capital of India. A cosmopolitan city in South Asia, New Delhi is home to a diverse population that speaks many different languages and holds a wide variety of religious beliefs. The city of New Delhi is part of the larger Delhi Metro Area whose population is over 32 million, and it is well-known for its chaotic traffic and large crowds.\n\nBut there’s more to New Delhi than massive crowds and traffic. In this post, we’ll explore some of the unique features of the capital city of India.\n\n## Where is New Delhi? [...] ### What was the first capital of India?\n\nBefore New Delhi was the capital of India, the city of Calcutta (Kolkata) became the nation’s capital in 1772. Various important statesmen and yogis have come from Calcutta, and the city was a center of culture and politics within India from the 18th century onward

## Initialize LLM

In [6]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

result = llm.invoke("Tell me a joke about AI.")
result

AIMessage(content='Why did the AI break up with its computer? It found someone who could handle its data better!', response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 14, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-8c8fb20a-bd05-4f60-af8f-9477032e9a3f-0')

## Create agent

In [14]:
# Setup Prompt
client = Client()
prompt = client.pull_prompt("hwchase17/react")

# Setup tools
tools = [search_tool]

In [16]:
# Create agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

## Agent executor

In [17]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

## Run

In [20]:
response = agent_executor.invoke(
    {
        "input": (
            "Tell me the latest news about India."
        )
    }
)



> Entering new AgentExecutor chain...
you should use the search engine to get the most up-to-date information.
Action: tavily_search_results_json
Action Input: India latest news[{'url': 'https://indianexpress.com/article/india/today-india-breaking-news-live-updates-8-august-2026-amarnath-yatra-suspended-assam-floods-us-tariff-russia-oil-10823520', 'content': 'India News Today Live Updates, 8 August 2026: Leader of Opposition in Lok Sabha addresses a “Chhatron Ki Goonj” meeting in Prayagraj today. Addressing the gathering of students, Gandhi says India has both data and youth and youth plus data is the 21st-century economy. Gandhi further said that AI has no meaning without your data.\n\n## If you are just joining us, this is The Indian Express Realtime breaking news blog – where we bring you the latest news updates of the day in real time. Scan the top headlines of the day here, or keep scrolling to see the latest updates stream in.\n\n## India stares at 100% US tariff threat after S

In [ ]:
print(response["output"])

The latest news about India includes updates on US tariff threats, protests in Jharkhand, and various other events happening in the country.
